# Arrays

In [1]:
from pathlib import Path
import yaml
from metasmith.python_api import *
from metasmith import examples

In [2]:
dtypes, contigs, references, transforms = examples.GenomicsAnnotation()
for x in [dtypes, contigs, references, transforms]:
    print(type(x))

<class 'metasmith.models.libraries.DataTypeLibrary'>
<class 'metasmith.models.libraries.DataInstanceLibrary'>
<class 'metasmith.models.libraries.DataInstanceLibrary'>
<class 'metasmith.models.libraries.TransformInstanceLibrary'>


In [3]:
# show contents of input XGDB
for x, d in contigs.manifest.items():
    s = 20-len(str(x))
    print(f"{x}{' '*s}({d})")

fosmid.fna          (genomics::contigs)


In [4]:
# show contents of reference XGDB (for databases, tool containers, other data dependencies, etc.)
for x, d in references.manifest.items():
    s = 20-len(str(x))
    print(f"{x}{' '*s}({d})")

blast.oci.uri       (genomics::oci_image_blast)
prodigal.oci.uri    (genomics::oci_image_prodigal)
swissprot_bcaa.faa  (genomics::protein_reference_fasta)


In [5]:
# show contents of transforms (available bioinformatics tools)
for x in transforms.manifest:
    print(x.stem)

blast
prodigal


In [6]:
path_to_agent_home = Path("./metasmith_home").resolve()
agent = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
if not path_to_agent_home.exists():
    agent.Deploy()
else:
    print("already deployed")

already deployed


In [7]:
# suppose we had 3 samples of the same type
# and we want to process them in parallel
SAMPLES = [f"{i+1:02}" for i in range(3)]

# we first create a new xgdb (DataInstanceLibrary) to hold the input array
path_to_array_xgdb = path_to_agent_home/"data/temp/contig_array.xgdb"
input_array = DataInstanceLibrary(path_to_array_xgdb)
# add the original data types to the new xgdb
for namespace, lib in contigs.types.items():
    input_array.AddTypeLibrary(namespace, lib)

# assuming the sample names are unique
# we can use the name as an additional property to specify each sample
# this way, each input is still a contig, but more specifically, the contigs for sample n
contig_properties = dtypes.types["contigs"].properties
samples = DataTypeLibrary(
    types = {sample_name: Endpoint(properties={sample_name}|contig_properties) for sample_name in SAMPLES},
)

# this registers the data type for each sample into the xgdb
input_array.AddTypeLibrary("sample", samples)

print("the original contig type")
print(yaml.safe_dump(dtypes["contigs"].Pack()))
print("sample 01")
print(yaml.safe_dump(samples["01"].Pack()))

the original contig type
properties:
  data: DNA sequence
  format: FASTA

sample 01
properties:
  _:
  - '01'
  data: DNA sequence
  format: FASTA



In [8]:
# we can now add each sample to the new data instance library
# by specifying the data type for each sample that includes the sample name
# for example "sample::01"
# since we don't have 3 different samples for this example,
# we can just copy the same contig file 3 times
template = contigs.location/"fosmid.fna"
input_array.Add(
    items = [
        (template, f"{template.stem}_{sample_name}.fna", f"sample::{sample_name}")
        for sample_name in SAMPLES
    ],
)

# the same as in the tutorial https://metasmith.readthedocs.io/en/latest/main/workflow.html
# we can give metasmith the new input_array xgdb
# and ask it to create annotations for each sample using the lineage constraint
task = agent.GenerateWorkflow(
    given=[input_array, references],
    transforms=[transforms],
    targets=[
        dtypes["orf_annotations"].WithLineage([samples[sample_name]]) # we want annotations for each sample
        for sample_name in SAMPLES
    ]
)

# this show the workflow plan
for step in task.plan.steps:
    print(f"step {step.order}: {step.transform.name}:{step.transform.model.key}")
    print(f"    uses: {[str(x.path)+':'+x.dtype.key for x in step.uses]}")
    print(f"   makes: {[str(x.path)+':'+x.dtype.key for x in step.produces]}")
    print()

step 1: prodigal:cS4YDXVi
    uses: ['fosmid_01.fna:STn6e2NR', 'prodigal.oci.uri:DzkAdyWQ']
   makes: ['orfs.faa:guo7ELq8']

step 2: blast:fUOlw7eC
    uses: ['orfs.faa:guo7ELq8', 'swissprot_bcaa.faa:fK7OFWQt', 'blast.oci.uri:EfZl0nW4']
   makes: ['annotations.csv:pBr8cKC1']

step 3: prodigal:cS4YDXVi
    uses: ['fosmid_02.fna:1hhMK6vs', 'prodigal.oci.uri:DzkAdyWQ']
   makes: ['orfs.faa:mwgZzhVz']

step 4: blast:fUOlw7eC
    uses: ['orfs.faa:mwgZzhVz', 'swissprot_bcaa.faa:fK7OFWQt', 'blast.oci.uri:EfZl0nW4']
   makes: ['annotations.csv:fEzCVcfl']

step 5: prodigal:cS4YDXVi
    uses: ['fosmid_03.fna:X1Y6DkLT', 'prodigal.oci.uri:DzkAdyWQ']
   makes: ['orfs.faa:M88j7mOZ']

step 6: blast:fUOlw7eC
    uses: ['orfs.faa:M88j7mOZ', 'swissprot_bcaa.faa:fK7OFWQt', 'blast.oci.uri:EfZl0nW4']
   makes: ['annotations.csv:VuLPavcc']



In [ ]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")

2025-05-05_00-47-34  | connecting to deployed agent
2025-05-05_00-47-34  | starting relay service
 | > 2025-05-05_00-47-34  | connecting to relay as [f5wLbhF2eSVH]


E| > 2025-05-05_00-47-34 E| relay server already running in [relay/connections]


2025-05-05_00-47-34 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/docs/metasmith_home/runs/KcFZuSHS]
2025-05-05_00-47-34 W| clearing previously staged task
2025-05-05_00-47-34  | sending metadata for workflow [KcFZuSHS]
2025-05-05_00-47-37  | staging
 | > including dev binds
 | > 2025-05-05_00-47-38  | api call to [stage_workflow] with [{'task_key': 'KcFZuSHS'}]
 | > 2025-05-05_00-47-38  | staging workflow [KcFZuSHS] with [2] data libs and [1] transform libs
 | > 2025-05-05_00-47-38  | ex| /home/tony/workspace/tools/Metasmith/main/docs/metasmith_home
 | > 2025-05-05_00-47-38  | work [/ws/runs/KcFZuSHS]
 | > 2025-05-05_00-47-38  | data [/msm_home/data]
 | > 2025-05-05_00-47-38  | external work [/home/tony/workspace/tools/Metasmith/main/docs/metasmith_home/runs/KcFZuSHS]
 | > 2025-05-05_00-47-38  | external data [/home/tony/workspace/tools/Metasmith/main/docs/metasmith_home/data]
 | > 2025-05-05_00-47-38  | moving remote data libraries to [/msm_home/data]
 | > 2025

In [ ]:
agent.RunWorkflow(task)

2025-05-05_00-47-38  | connecting to deployed agent
2025-05-05_00-47-39  | starting relay service
 | > 2025-05-05_00-47-39  | connecting to relay as [pJGHlsvBpgoH]


E| > 2025-05-05_00-47-39 E| relay server already running in [relay/connections]


2025-05-05_00-47-39  | triggering execution of [KcFZuSHS]
2025-05-05_00-47-39  | closing connection


In [40]:
agent.CheckWorkflow(task)

2025-05-05_01-02-34  | connecting to deployed agent
2025-05-05_01-02-34  | starting relay service
 | > 2025-05-05_01-02-35  | connecting to relay as [sV8jHUYR9M3H]


E| > 2025-05-05_01-02-35 E| relay server already running in [relay/connections]


 | > including dev binds
 | > 2025-05-05_01-02-36  | api call to [check_workflow] with [{'key': 'KcFZuSHS'}]
 | > 2025-05-05_01-02-36  | searching for logs
 | > 2025-05-05_01-02-36  | found [1] runs
 | > 2025-05-05_01-02-36  |     1: [logs.2025-05-05_00-47-39]
 | > 2025-05-05_01-02-36  | here is the main log of the latest run [logs.2025-05-05_00-47-39]
 | > 2025-05-05_01-02-36  | >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
 | > 2025-05-05_01-02-36  | 
 | > including dev binds
 | > 2025-05-05_00-47-40  | api call to [run_workflow] with [{'key': 'KcFZuSHS', 'log_dir': '_metasmith/logs.2025-05-05_00-47-39'}]
 | > 2025-05-05_00-47-40  | start time [2025-05-05_00-47-40]
 | > 2025-05-05_00-47-40  | running workflow [KcFZuSHS] with preset [default]
 | > 2025-05-05_00-47-40  | loading agent metadata
 | > 2025-05-05_00-47-40  | workspace [/msm_home/runs/KcFZuSHS]
 | > 2025-05-05_00-47-40  | external workspace [/home/tony/workspace/tools/Metasmith/main/docs/metasmith_home/r

In [ ]:
# the printout above should show the path to results
results_path = agent.home.GetPath()/f"runs/{task.plan._key}/results"
results = DataInstanceLibrary.Load(results_path)

# show the results
for path, dtype_name, dtype in results.Iterate():
    parents = results.parents.get(path, [])
    print(f"{dtype_name} at <results>/{path} was made from {[p.name for p in parents]}")

genomics::orf_annotations at <results>/0002/annotations.csv was made from ['sample::01']
genomics::orf_annotations at <results>/0004/annotations.csv was made from ['sample::02']
genomics::orf_annotations at <results>/0006/annotations.csv was made from ['sample::03']
